# RAR Per-Seed Campaign — Clean Runner

Runs the RAR experiment **one seed at a time**. Each seed writes its own
`pilot_seed_<seed>.json`, so a Colab disconnect never wipes finished work.

**Key handling:** loads automatically from **Colab Secrets** (the 🔑 icon).
Secret name must be `OPENROUTER_API_KEY` with *Notebook access* ON.

**Drive backup is optional** — if mounting fails, the run continues on local disk.

**Safety guard:** if the key is ever missing, the code *raises* instead of
silently fabricating fake results.

Run cells top to bottom. Don't start the seed loop until **PRE-FLIGHT** passes.

## 1 · Clone repo & install deps

In [ ]:
import os

REPO_DIR = '/content/recursive-autonomy-research'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tahir-yamin/recursive-autonomy-research.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!pip install -q torch numpy scipy scikit-learn networkx aiohttp matplotlib

## 2 · Load API key from Colab Secrets

One-time setup: 🔑 icon → **+ Add new secret** → name `OPENROUTER_API_KEY`,
paste your key, toggle **Notebook access** ON. Then just run this cell.

In [ ]:
import os

key = None
try:
    from google.colab import userdata
    key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    print('Could not read Colab Secret:', e)

if not key:
    raise ValueError(
        '\n>>> No key found. Open the 🔑 panel, add OPENROUTER_API_KEY, '
        'and turn ON Notebook access for it, then re-run this cell.'
    )

os.environ['OPENROUTER_API_KEY'] = key
os.environ['OPENROUTER_MODEL']   = 'google/gemma-3-27b-it:free'
os.environ['RAR_CYCLES']         = '60'
print(f'Key loaded (ends ...{key[-6:]})')
print('Model:', os.environ['OPENROUTER_MODEL'])

## 3 · (Optional) Mount Google Drive for backup

If mounting fails, this cell just warns and the run continues on local disk.

In [ ]:
DRIVE_BACKUP = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BACKUP = '/content/drive/MyDrive/RAR_results'
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print('Drive backup ON ->', DRIVE_BACKUP)
except Exception as e:
    DRIVE_BACKUP = None
    print('Drive mount skipped (continuing on local disk only):', e)

## 4 · PRE-FLIGHT — one real API call

Proves the key **and** the Gemma model respond before a multi-hour run.
**Do not proceed if this fails.**

In [ ]:
import sys, os
os.chdir('/content/recursive-autonomy-research')
if 'run_pilot_experiment' in sys.modules:
    del sys.modules['run_pilot_experiment']
import run_pilot_experiment as rpe

resp = await rpe.call_llm('Reply with the single word: OK')
if not resp:
    raise RuntimeError(
        '>>> PRE-FLIGHT FAILED: empty response. Check key validity, model name, '
        'and free-tier quota. Do NOT start the seed loop.'
    )
print('PRE-FLIGHT OK — model said:', repr(resp[:80]))

## 5 · Run seeds (one at a time)

Already-completed seeds are auto-skipped. Each finished seed is backed up to
Drive if available. Safe to re-run after a disconnect — it resumes.

In [ ]:
import os, sys, shutil
os.chdir('/content/recursive-autonomy-research')

seeds_to_run = [42, 7, 13, 23, 88, 99, 101, 107, 113, 127]

if not os.environ.get('OPENROUTER_API_KEY'):
    raise ValueError('Key missing — re-run cell 2.')

for mod in ('run_pilot_experiment', 'run_deep_learning_harness'):
    if mod in sys.modules:
        del sys.modules[mod]
import run_pilot_experiment as rpe

# restore any seed files already backed up to Drive
if DRIVE_BACKUP and os.path.exists(DRIVE_BACKUP):
    import glob
    for f in glob.glob(os.path.join(DRIVE_BACKUP, 'pilot_seed_*.json')):
        dst = os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy2(f, dst)
            print('Restored', dst, 'from Drive')

for seed in seeds_to_run:
    out_file = f'pilot_seed_{seed}.json'
    if os.path.exists(out_file):
        print(f'Seed {seed}: already done, skipping.')
        continue
    print(f'\n===== STARTING SEED {seed} =====')
    os.environ['RAR_OUTPUT_FILE'] = out_file
    rpe.SEEDS = [seed]
    rpe.CYCLES = int(os.environ.get('RAR_CYCLES', '60'))
    await rpe.execute_campaign()
    if os.path.exists(out_file):
        print(f'Seed {seed}: DONE -> {out_file} ({os.path.getsize(out_file):,} bytes)')
        if DRIVE_BACKUP:
            shutil.copy2(out_file, os.path.join(DRIVE_BACKUP, out_file))
            print('   backed up to Drive')
    else:
        print(f'WARNING: seed {seed} produced no file!')

print('\n===== ALL REQUESTED SEEDS COMPLETE =====')

## 6 · Merge seeds → final pilot_results.json (real Wilcoxon p)

In [ ]:
import os, glob, shutil
os.chdir('/content/recursive-autonomy-research')

if DRIVE_BACKUP and os.path.exists(DRIVE_BACKUP):
    for f in glob.glob(os.path.join(DRIVE_BACKUP, 'pilot_seed_*.json')):
        dst = os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy2(f, dst)

seed_files = sorted(glob.glob('pilot_seed_*.json'))
print(f'{len(seed_files)} seed files:', [os.path.basename(f) for f in seed_files])

if len(seed_files) < 2:
    print('Need >=2 seeds to merge.')
else:
    !python merge_seeds.py
    if DRIVE_BACKUP and os.path.exists('pilot_results.json'):
        shutil.copy2('pilot_results.json', os.path.join(DRIVE_BACKUP, 'pilot_results.json'))
        print('Merged result backed up to Drive')

## 7 · Sanity check — the honest numbers

In [ ]:
import json
with open('pilot_results.json') as f:
    r = json.load(f)
print('Seeds :', r['SEEDS'])
print('Cycles:', r['CYCLES'])
print('Wilcoxon p (RAR > Baseline):', r['wilcoxon_p_value_RAR_vs_Baseline'])
print()
for cond in ['stateless_baseline', 'vector_rag', 'rar_compressed']:
    d = r['data']['conditions'][cond]
    ta = d['test_accuracies']; nt = d['net_tokens']
    print(f'{cond:20s} test_acc={sum(ta)/len(ta):.4f}  net_tokens={sum(nt)/len(nt):,.0f}')